# Unidad 3: Procesos de Extracción, Transformación y Carga (ETL)
### Procesamiento de Datos (Nivel 4)
**Carrera de Ciencia de Datos** — Modalidad En Línea
**Docente:** Galo Valverde

---

> Los procesos ETL transforman datos en bruto en activos de conocimiento estratégico.

Esta unidad final unifica los conceptos de almacenamiento y manipulación dentro del flujo maestro
de la ingeniería de datos corporativa: **Integración de Datos**, operaciones avanzadas de combinación
en Pandas (`concat`, `merge`, `groupby`), las fases **Extract–Transform–Load**, SQL avanzado
(vistas, agregaciones, optimización de consultas), automatización con Python, orquestación mediante
DAGs, y optimización de rendimiento con vectorización y cómputo distribuido.

**Logros de aprendizaje:**
1. Diseñar flujos ETL multi-fuentes.
2. Escribir SQL avanzado de transformación.
3. Automatizar procesos con scripts de Python.
4. Optimizar el rendimiento de procesamiento.


In [1]:
# Dependencias de la unidad
import pandas as pd
import numpy as np
import sqlite3
import datetime
import time

pd.set_option("display.max_columns", None)
np.random.seed(42)
print("Entorno listo.")


Entorno listo.


## 1. Integración de Datos

La integración de datos unifica fuentes heterogéneas para mitigar el caos: la consolidación de
fuentes implica resolver la heterogeneidad semántica y estructural de los datos empresariales que
residen en repositorios diversos.

| Reto | Descripción |
|---|---|
| **Consolidación física** | Extraer datos de múltiples bases transaccionales, limpiarlos y almacenarlos por adelantado en un sitio unificado (Data Warehouse). Enfoque *update-driven* que maximiza el rendimiento analítico sin interferir con las operaciones diarias. |
| **Problema de identificación** | Ocurre al casar entidades equivalentes en esquemas heterogéneos (ej. `customer_id` vs. `cust_num`). Requiere análisis de esquemas lógicos y metadatos para evitar duplicados redundantes. |
| **Resolución de conflictos** | Unificar diferencias de escala, codificación y representación física (ej. kilos vs. libras, distintas monedas). Se resuelve en la fase de mapeo lógico previo a la persistencia. |

### Calidad de datos en el contexto de integración (ISO 8000)
- **Exactitud** — los datos reflejan correctamente la realidad del minimundo.
- **Completitud** — identificar valores faltantes y definir políticas de imputación.
- **Consistencia** — resolver inconsistencias por duplicación o desnormalización.
- **Vigencia** — la información está disponible dentro de la ventana útil para decidir.

### El proceso de depuración de datos
1. **Perfilamiento** — estadísticas descriptivas, cardinalidad, valores nulos.
2. **Detección de discrepancias** — reglas de negocio para señalar inconsistencias y outliers.
3. **Transformación correctiva** — mapeos automáticos (Pandas/Python) para corregir formato.
4. **Auditoría y control** — monitoreo constante de métricas de calidad en el Data Warehouse.


### Gestión de Metadatos

Los metadatos son "datos acerca de los datos": el mapa indispensable para operar, auditar y
gobernar un pipeline analítico.

| Tipo | Contenido |
|---|---|
| **Estructurales** | Esquemas de tablas relacionales (Star, Snowflake), dimensiones, hechos, cubos OLAP, jerarquías de granularidad |
| **Operacionales** | Fechas de ejecución, volumen de registros, linaje del dato, logs de errores/accesos/tiempos |
| **De mapeo** | Reglas y fórmulas de transformación de campos, equivalencia de tipos entre fuentes, políticas de limpieza/archivado |


## 2. Operaciones Avanzadas de Combinación en Pandas

### 2.1 Concatenación (`pd.concat`)
Apila conjuntos de datos de forma matricial. Alinea automáticamente etiquetas de filas/columnas;
si no hay solapamiento, introduce `NaN`. Se puede forzar intersección (`join='inner'`) o unión
completa (`join='outer'`).


In [2]:
# Concatenación: alineación de índices
df1 = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
df2 = pd.DataFrame({"a": [5, 6], "c": [7, 8]})

res_row = pd.concat([df1, df2], axis=0, ignore_index=True)   # apilado por filas (outer por defecto)
res_col = pd.concat([df1, df2], axis=1)                       # apilado por columnas
res_inner = pd.concat([df1, df2], join="inner")                # solo columnas en común

print("--- Concatenación por filas (outer, con NaN) ---")
print(res_row)
print("\n--- Concatenación por columnas ---")
print(res_col)
print("\n--- Concatenación con intersección (inner) ---")
print(res_inner)


--- Concatenación por filas (outer, con NaN) ---
   a    b    c
0  1  3.0  NaN
1  2  4.0  NaN
2  5  NaN  7.0
3  6  NaN  8.0

--- Concatenación por columnas ---
   a  b  a  c
0  1  3  5  7
1  2  4  6  8

--- Concatenación con intersección (inner) ---
   a
0  1
1  2
0  5
1  6


### 2.2 Combinación basada en claves (`pd.merge`)

`pd.merge()` es el motor principal para cruces de datos (*joins*) basados en columnas clave,
similar a SQL:

| `how` | Comportamiento |
|---|---|
| `inner` | Solo las claves presentes en ambas fuentes (intersección) |
| `left`  | Preserva las claves del DataFrame izquierdo; nulos si el derecho no coincide |
| `right` | Preserva las claves del DataFrame derecho |
| `outer` | Unión completa de todas las claves |

`df.join()` es una función de conveniencia optimizada para cruzar por el índice de fila.


In [3]:
# Merge por claves distintas (equijoin) entre tabla de hechos y dimensión
empleados = pd.DataFrame({
    "dni": ["e1", "e2", "e3"],
    "nombre": ["José", "Alberto", "Alicia"],
    "dno": [5, 4, 5]
})
departamentos = pd.DataFrame({
    "num_dpto": [4, 5],
    "nombre_dpto": ["Admin", "Investigación"]
})

resultado_inner = pd.merge(empleados, departamentos, left_on="dno", right_on="num_dpto", how="inner")
resultado_left = pd.merge(empleados, departamentos, left_on="dno", right_on="num_dpto", how="left")

print("--- INNER JOIN ---")
print(resultado_inner)
print("\n--- LEFT JOIN (preserva todos los empleados) ---")
print(resultado_left)


--- INNER JOIN ---
  dni   nombre  dno  num_dpto    nombre_dpto
0  e1     José    5         5  Investigación
1  e2  Alberto    4         4          Admin
2  e3   Alicia    5         5  Investigación

--- LEFT JOIN (preserva todos los empleados) ---
  dni   nombre  dno  num_dpto    nombre_dpto
0  e1     José    5         5  Investigación
1  e2  Alberto    4         4          Admin
2  e3   Alicia    5         5  Investigación


### 2.3 Agregaciones: paradigma Split-Apply-Combine (`groupby`)

Flujo analítico de Hadley Wickham en tres fases:
1. **Split** — segmentar el DataFrame en grupos lógicos según una o más claves.
2. **Apply** — ejecutar funciones agregadas/estadísticas dentro de cada grupo.
3. **Combine** — consolidar los resultados en una estructura unificada.


In [4]:
# groupby simple y agrupamientos múltiples con funciones diversas
ventas = pd.DataFrame({
    "sucursal": ["A", "A", "B", "B", "C", "C"],
    "categoria": ["PC", "TV", "PC", "PC", "TV", "TV"],
    "monto": [1200, 800, 1500, 1100, 950, 850]
})

resumen = ventas.groupby("sucursal")["monto"].sum()
print("--- Resumen por sucursal ---")
print(resumen)

res_multi = ventas.groupby(["sucursal", "categoria"]).agg({"monto": ["sum", "mean", "count"]})
print("\n--- Agrupamiento múltiple (sucursal x categoría) ---")
print(res_multi)


--- Resumen por sucursal ---
sucursal
A    2000
B    2600
C    1800
Name: monto, dtype: int64

--- Agrupamiento múltiple (sucursal x categoría) ---
                   monto              
                     sum    mean count
sucursal categoria                    
A        PC         1200  1200.0     1
         TV          800   800.0     1
B        PC         2600  1300.0     2
C        TV         1800   900.0     2


## 3. Las Fases del Proceso ETL

### 3.1 Extracción (E)
Objetivo: leer de forma eficiente y segura la información relevante de múltiples fuentes
transaccionales, minimizando el impacto en los servidores operativos.

**Técnicas de Change Data Capture (CDC):**
- **Basada en consultas** — timestamps o IDs incrementales para extraer solo lo nuevo/actualizado.
- **Basada en triggers** — disparadores que registran cambios en una tabla de auditoría.
- **Basada en logs** — lectura directa de logs de transacciones del SGBD (ej. WAL en PostgreSQL).

**Tipología de fuentes:**
- Estructuradas (SGBD relacionales vía SQL)
- Semiestructuradas (JSON, XML, Parquet, REST APIs)
- No estructuradas (texto plano, páginas web vía scraping)

### 3.2 Transformación (T)
Núcleo computacional del ETL: estructura los datos capturados para cumplir el modelo de destino.

| Etapa | Acciones |
|---|---|
| **Depuración** | Imputación de nulos, eliminación de ruido y duplicados |
| **Formateo** | Fechas a ISO 8601, unificación de strings/UTF-8, escalamiento de unidades |
| **Enriquecimiento** | Campos derivados, indicadores binarios, *lookup* de variables con metadatos |
| **Agregación** | Resumen/conteo de transacciones, reducción de granularidad, pivotado lógico |

### 3.3 Carga (L)
Persistencia física en sistemas optimizados para lectura concurrente (Data Warehouse/Data Lake).

- **Carga inicial** — puebla estructuras vacías por primera vez.
- **Carga incremental** — inserta/actualiza solo cambios nuevos capturados por la extracción.

**Buenas prácticas:** desactivar índices temporalmente en *bulk loads*, validar integridad
referencial y llaves foráneas, y usar transacciones ACID (*two-phase commit*) para evitar
estados parciales corruptos.


In [5]:
# Simulación completa de un mini-pipeline ETL con dos fuentes SQLite (Extract -> Transform -> Load)

# --- Preparar fuentes operacionales (simulan sistemas heterogéneos) ---
conn_ventas = sqlite3.connect(":memory:")
conn_clientes = sqlite3.connect(":memory:")

pd.DataFrame({
    "id_venta": range(1, 9),
    "cliente_id": [1, 2, 1, 3, 2, 4, 3, 1],
    "monto": [120.5, None, 340.0, 90.0, 15000.0, 60.25, None, 210.0],  # incluye nulo y outlier
    "fecha": ["2026-01-05", "2026-01-06", "2026-01-06", "2026-01-07",
              "2026-01-08", "2026-01-08", "2026-01-09", "2026-01-10"]
}).to_sql("ventas", conn_ventas, index=False, if_exists="replace")

pd.DataFrame({
    "cliente_id": [1, 2, 3, 4],
    "nombre": ["Ana Torres", "Luis Vera", "Maria Paz", "Kevin Ruiz"],
    "ciudad": ["Guayaquil", "Quito", "Cuenca", "Guayaquil"]
}).to_sql("clientes", conn_clientes, index=False, if_exists="replace")

print("Fuentes operacionales creadas: 'ventas' y 'clientes' (simulando dos SGBD distintos).")


Fuentes operacionales creadas: 'ventas' y 'clientes' (simulando dos SGBD distintos).


In [6]:
# --- EXTRACT: leer de ambas fuentes heterogéneas ---
df_ventas = pd.read_sql("SELECT * FROM ventas", conn_ventas)
df_clientes = pd.read_sql("SELECT * FROM clientes", conn_clientes)

print("Ventas extraídas:")
print(df_ventas)
print("\nClientes extraídos:")
print(df_clientes)


Ventas extraídas:
   id_venta  cliente_id     monto       fecha
0         1           1    120.50  2026-01-05
1         2           2       NaN  2026-01-06
2         3           1    340.00  2026-01-06
3         4           3     90.00  2026-01-07
4         5           2  15000.00  2026-01-08
5         6           4     60.25  2026-01-08
6         7           3       NaN  2026-01-09
7         8           1    210.00  2026-01-10

Clientes extraídos:
   cliente_id      nombre     ciudad
0           1  Ana Torres  Guayaquil
1           2   Luis Vera      Quito
2           3   Maria Paz     Cuenca
3           4  Kevin Ruiz  Guayaquil


In [7]:
# --- TRANSFORM: depuración, formateo, enriquecimiento e integración ---

# 1. Depuración: imputar nulos con la mediana
df_ventas["monto"] = df_ventas["monto"].fillna(df_ventas["monto"].median())

# 2. Formateo: fechas a ISO 8601 (datetime real)
df_ventas["fecha"] = pd.to_datetime(df_ventas["fecha"])

# 3. Detección de outliers (regla de negocio simple) y corrección
limite_superior = df_ventas["monto"].quantile(0.75) + 1.5 * (
    df_ventas["monto"].quantile(0.75) - df_ventas["monto"].quantile(0.25)
)
df_ventas.loc[df_ventas["monto"] > limite_superior, "monto"] = df_ventas["monto"].median()

# 4. Enriquecimiento: integración con clientes (resolución de heterogeneidad de fuentes)
df_integrado = pd.merge(df_ventas, df_clientes, on="cliente_id", how="left")

# 5. Enriquecimiento: campo derivado + indicador binario
df_integrado["categoria_monto"] = np.where(df_integrado["monto"] > 100, "Alta", "Baja")

print("Dataset transformado e integrado:")
df_integrado


Dataset transformado e integrado:


,id_venta,cliente_id,monto,fecha,nombre,ciudad,categoria_monto
0,1,1,120.50,2026-01-05,Ana Torres,Guayaquil,Alta
1,2,2,165.25,2026-01-06,Luis Vera,Quito,Alta
2,3,1,340.00,2026-01-06,Ana Torres,Guayaquil,Alta
3,4,3,90.00,2026-01-07,Maria Paz,Cuenca,Baja
4,5,2,165.25,2026-01-08,Luis Vera,Quito,Alta
5,6,4,60.25,2026-01-08,Kevin Ruiz,Guayaquil,Baja
6,7,3,165.25,2026-01-09,Maria Paz,Cuenca,Alta
7,8,1,210.00,2026-01-10,Ana Torres,Guayaquil,Alta


In [8]:
# --- LOAD: persistir en un Data Warehouse (carga inicial) ---
warehouse = sqlite3.connect(":memory:")
df_integrado.to_sql("hechos_ventas", warehouse, index=False, if_exists="replace")

print("Job ETL exitoso:", datetime.datetime.now())
print("\nVerificación de la carga en el Data Warehouse:")
pd.read_sql("SELECT * FROM hechos_ventas LIMIT 5", warehouse)


Job ETL exitoso: 2026-09-01 04:17:28.779154

Verificación de la carga en el Data Warehouse:


,id_venta,cliente_id,monto,fecha,nombre,ciudad,categoria_monto
0,1,1,120.50,2026-01-05 00:00:00,Ana Torres,Guayaquil,Alta
1,2,2,165.25,2026-01-06 00:00:00,Luis Vera,Quito,Alta
2,3,1,340.00,2026-01-06 00:00:00,Ana Torres,Guayaquil,Alta
3,4,3,90.00,2026-01-07 00:00:00,Maria Paz,Cuenca,Baja
4,5,2,165.25,2026-01-08 00:00:00,Luis Vera,Quito,Alta


## 4. ETL vs. ELT

Las arquitecturas tradicionales ETL procesan los datos en un servidor de *staging* dedicado antes
de cargarlos. Con los Data Lakes elásticos modernos, el modelo **ELT** transforma los datos
directamente dentro del repositorio de destino.

| Característica | ETL Tradicional (staging server) | ELT Moderno (nube / Data Lake) |
|---|---|---|
| **Secuencia lógica** | Extracción → Transformación → Carga en BD | Extracción → Carga directa → Transformación *in-situ* |
| **Sitio de cómputo** | Servidor de staging dedicado | Sistemas nativos del Data Warehouse/Lake destino |
| **Volumen soportado** | Limitado por CPU/RAM del staging server | Escalable a petabytes (cómputo elástico) |
| **Carga inicial** | Lenta (preprocesamiento previo) | Extremadamente rápida (datos crudos directos) |
| **Casos de uso** | BD estructuradas, integración on-prem | Ingesta masiva de Big Data en la nube (Snowflake, Databricks) |


## 5. SQL como Base Matemática de la Manipulación Analítica

Las consultas SQL operan sobre álgebra relacional y cálculo de predicados de primer orden:

- **Selección (σ)** — filtrado de filas (`WHERE`)
- **Proyección (π)** — selección de columnas (`SELECT`)
- **Concatenación (⋈)** — combinación de tuplas por atributos comunes (`JOIN`)

A continuación replicamos sobre nuestro Data Warehouse en memoria: una **vista** consolidada
y **funciones de agregación** con `GROUP BY`, tal como se describe en las diapositivas.


In [9]:
# Vista lógica consolidada + agregación GROUP BY sobre el Data Warehouse cargado
cur = warehouse.cursor()

cur.execute('''
CREATE VIEW IF NOT EXISTS resumen_ciudad AS
SELECT
    ciudad,
    COUNT(id_venta)   AS total_ventas,
    SUM(monto)        AS monto_total,
    AVG(monto)        AS monto_promedio
FROM hechos_ventas
GROUP BY ciudad
''')

resumen_ciudad = pd.read_sql("SELECT * FROM resumen_ciudad ORDER BY monto_total DESC", warehouse)
resumen_ciudad


,ciudad,total_ventas,monto_total,monto_promedio
0,Guayaquil,4,730.75,182.6875
1,Quito,2,330.50,165.2500
2,Cuenca,2,255.25,127.6250


In [10]:
# Subconsulta correlacionada (patrón NOT EXISTS anidado)
# Ejemplo: clientes cuyas compras SIEMPRE superaron el promedio general (ningún registro por debajo)
promedio_general = pd.read_sql("SELECT AVG(monto) AS prom FROM hechos_ventas", warehouse)["prom"].iloc[0]

query = f'''
SELECT DISTINCT nombre, ciudad
FROM hechos_ventas c
WHERE NOT EXISTS (
    SELECT 1 FROM hechos_ventas h
    WHERE h.cliente_id = c.cliente_id
    AND h.monto <= {promedio_general}
)
'''
pd.read_sql(query, warehouse)


,nombre,ciudad
0,Luis Vera,Quito


### Optimización de consultas SQL (SQL Tuning)

Un cuello de botella crítico es ejecutar transformaciones masivas ineficientes que provoquen
recorridos completos de tabla (*Full Table Scans*).

- **Índices apropiados** — sobre columnas de `ORDER BY` y `WHERE` frecuentes.
- **Evitar operadores costosos** — reemplazar `NOT IN`/subconsultas complejas por `LEFT JOIN` con
  comprobación de nulos cuando sea posible.
- **Revisar el plan de ejecución (`EXPLAIN`)** — inspeccionar el costo estimado por el SGBD.
- **Índice de combinación (Join Index)** — pre-registra emparejamientos entre hechos y dimensiones.
- **Índices de mapa de bits** — eficientes para columnas categóricas de baja cardinalidad.
- **Proyecciones tempranas** — evitar `SELECT *`; pedir solo las columnas necesarias.


In [11]:
# EXPLAIN QUERY PLAN en SQLite: inspeccionar el costo antes de ejecutar
plan = pd.read_sql(
    "EXPLAIN QUERY PLAN SELECT ciudad, SUM(monto) FROM hechos_ventas GROUP BY ciudad",
    warehouse
)
plan


,id,parent,notused,detail
0,6,0,0,SCAN hechos_ventas
1,8,0,0,USE TEMP B-TREE FOR GROUP BY


## 6. Automatización con Python

Un pipeline analítico debe correr de forma automatizada y periódica sin intervención humana.

- **Windows Task Scheduler** — ejecución basada en disparadores temporales.
- **Cron Jobs (Linux/Unix)** — sintaxis clásica `min hora día mes día-semana`.

Python actúa como lenguaje de pegamento para coordinar SGBD relacionales, NoSQL documentales
y almacenamiento local, tal como en el Job de la diapositiva 16.


In [12]:
# Job de automatización ETL completo, con manejo de excepciones
def run_etl_job(conn_origen, conn_destino):
    try:
        # 1. Extraer de la base de datos origen
        df = pd.read_sql("SELECT * FROM ventas", conn_origen)

        # 2. Transformar nulos y fechas
        df["monto"] = df["monto"].fillna(0)
        df["fecha"] = pd.to_datetime(df["fecha"])

        # 3. Cargar en el Data Warehouse analítico
        df.to_sql("hechos_ventas_job", conn_destino, if_exists="replace", index=False)
        print("Job ETL exitoso:", datetime.datetime.now())
        return True
    except Exception as e:
        print("Error en Job:", e)
        return False

exito = run_etl_job(conn_ventas, warehouse)
print("¿Ejecución exitosa?:", exito)


Job ETL exitoso: 2026-09-01 04:17:28.824214
¿Ejecución exitosa?: True


## 7. Orquestación de Pipelines (DAGs)

Un pipeline moderno no es un solo script, sino una red de tareas interconectadas con
dependencias lógicas: un **DAG** (*Directed Acyclic Graph*):

- **Dirigido** — tiene un sentido de flujo claro (una tarea precede a otra).
- **Acíclico** — no contiene ciclos infinitos de ejecución.

```
Tarea 1: Extracción (Consultar APIs y SGBDs)
        ↓
Tarea 2: Transformación (Limpieza, normalización)
        ↓
Tarea 3: Carga (Persistir en Data Warehouse)
```

**Monitoreo:** logs persistentes para depuración asíncrona y alertas automáticas (Slack, correo)
ante fallos. Orquestadores modernos: **Apache Airflow**, **Prefect**.


In [13]:
# Simulación mínima de un DAG de 3 tareas con manejo de dependencias y logging
def tarea_extraccion():
    print("[1/3] Extracción: consultando fuentes...")
    return pd.read_sql("SELECT * FROM ventas", conn_ventas)

def tarea_transformacion(df):
    print("[2/3] Transformación: limpiando y normalizando...")
    df = df.copy()
    df["monto"] = df["monto"].fillna(df["monto"].median())
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df

def tarea_carga(df):
    print("[3/3] Carga: persistiendo en el Data Warehouse...")
    df.to_sql("hechos_ventas_dag", warehouse, if_exists="replace", index=False)
    return True

def ejecutar_dag():
    log = []
    try:
        df_extraido = tarea_extraccion()
        log.append(("extraccion", "OK", len(df_extraido)))

        df_transformado = tarea_transformacion(df_extraido)
        log.append(("transformacion", "OK", len(df_transformado)))

        exito = tarea_carga(df_transformado)
        log.append(("carga", "OK" if exito else "FALLO", None))

        print("\nDAG ejecutado exitosamente:", datetime.datetime.now())
    except Exception as e:
        log.append(("error", str(e), None))
        print("Alerta: fallo en el DAG ->", e)
    return log

registro_ejecucion = ejecutar_dag()
pd.DataFrame(registro_ejecucion, columns=["tarea", "estado", "registros"])


[1/3] Extracción: consultando fuentes...
[2/3] Transformación: limpiando y normalizando...
[3/3] Carga: persistiendo en el Data Warehouse...

DAG ejecutado exitosamente: 2026-09-01 04:17:28.835615


,tarea,estado,registros
0,extraccion,OK,8.0
1,transformacion,OK,8.0
2,carga,OK,NaN


## 8. Vectorización: Optimización de Rendimiento

Un antipatrón dañino en Ciencia de Datos es procesar DataFrames extensos con bucles `for`
iterativos en Python:

- Python es interpretado con tipado dinámico → sobrecarga en cada iteración.
- No aprovecha la caché del procesador por dispersión de objetos en RAM.

**Solución:** operaciones vectorizadas en NumPy/Pandas, que delegan el cálculo a bloques
optimizados escritos en C.


In [14]:
# Comparación real de rendimiento: bucle iterativo vs. vectorización
n = 200_000
df_bench = pd.DataFrame({"monto": np.random.uniform(10, 5000, n)})

# ANTIPATRÓN: bucle iterativo con .iterrows()
inicio = time.time()
montos_pequenos = df_bench.head(5000)  # reducimos tamaño: iterrows es extremadamente lento
impuesto_lento = []
for idx, row in montos_pequenos.iterrows():
    impuesto_lento.append(row["monto"] * 0.15)
tiempo_lento = time.time() - inicio

# MEJOR PRÁCTICA: operación vectorizada sobre el dataset completo
inicio = time.time()
df_bench["impuesto"] = df_bench["monto"] * 0.15
tiempo_vectorizado = time.time() - inicio

print(f"Bucle iterativo (5,000 filas):      {tiempo_lento:.4f} s")
print(f"Vectorizado ({n:,} filas):        {tiempo_vectorizado:.4f} s")
print(f"\n-> La versión vectorizada procesó {n // 5000}x más filas en una fracción del tiempo.")


Bucle iterativo (5,000 filas):      0.0629 s
Vectorizado (200,000 filas):        0.0018 s

-> La versión vectorizada procesó 40x más filas en una fracción del tiempo.


In [15]:
# Selección condicional vectorizada con np.where (evita apply/loops)
df_bench["categoria"] = np.where(df_bench["monto"] > 1000, "Alta", "Baja")
df_bench["categoria"].value_counts()


categoria
Alta    160361
Baja     39639
Name: count, dtype: int64

## 9. Paralelización y Frameworks Distribuidos

Cuando el volumen de datos excede la memoria RAM de un solo servidor (Terabytes/Petabytes),
se recurre a la computación distribuida:

- **MapReduce (Hadoop)** — divide el procesamiento en `Map` (distribuir carga entre nodos) y
  `Reduce` (agrupar y consolidar). Escribe intensivamente en disco.
- **Apache Spark** — procesa *in-memory* con RDDs y DataFrames distribuidos; hasta 100x más
  rápido que Hadoop para cargas analíticas.

**Técnicas de paralelización física:**
- **SMP** (Symmetric Multi-Processing) — memoria compartida, escala con hilos locales del CPU.
- **MPP** (Massively Parallel Processing) — múltiples servidores cooperando por red de interconexión.
- **Spark Streaming** — unifica cálculo *batch* con ingesta en tiempo real de flujos infinitos.


## 10. Reproducibilidad y Documentación

| Pilar | Prácticas |
|---|---|
| **Minimizar redundancia** | Dimensiones compartidas (*conformed dimensions*), convenciones estrictas de nombres |
| **Reproducibilidad** | Control de versiones (Git/GitHub), entornos aislados y reproducibles (Docker) |
| **Documentación** | Diccionarios de datos actualizados, documentación técnica del pipeline, comentarios claros |
| **Monitoreo** | Registro sistemático de tiempos de ejecución, alertas tempranas ante excepciones críticas |


## Conclusiones

- ETL (o ELT en arquitecturas modernas en la nube) es el flujo maestro que convierte datos
  dispersos y heterogéneos en activos de conocimiento estratégico.
- Las operaciones de Pandas (`concat`, `merge`, `groupby`) son la base práctica para integrar
  y resumir información corporativa.
- SQL sigue siendo el lenguaje matemático fundamental para transformar datos directamente en
  el motor de persistencia, con vistas, agregaciones y optimización de consultas.
- La automatización (scripts Python, cron/Task Scheduler) y la orquestación (DAGs) garantizan
  que los pipelines corran de forma confiable y sin intervención manual.
- La vectorización y el cómputo distribuido (Spark, MapReduce) son indispensables cuando el
  volumen de datos crece más allá de la capacidad de un solo proceso.
- La reproducibilidad y documentación rigurosa cierran el ciclo completo de la ingeniería de
  datos: desde la captura hasta la analítica avanzada.

---

## Referencias y Enlaces de Consulta

**Enlaces oficiales:**
- Optimización de Consultas SQL (Oracle): https://docs.oracle.com/en/database/oracle/oracle-database/19/tgsql/index.html
- Arquitectura de Datos Distribuidos en Apache Spark: https://spark.apache.org/docs/latest/index.html

**Referencias bibliográficas (APA):**
- Casas-Roma, J., Nin Guerrero, J., & Julbe López, F. (2019). *Big Data. Análisis de datos en entornos masivos* (1ª ed.). Editorial UOC.
- Connolly, T. M., & Begg, C. E. (2005). *Sistemas de Bases de Datos: Un enfoque práctico para diseño, implementación y gestión* (4ª ed.). Pearson/Addison-Wesley.
- Kleppmann, M. (2023). *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems*. O'Reilly Media.
